# Network of Thrones: hybrid RAG + knowledge graph assistant

This notebook combines two complementary knowledge sources:

- **Unstructured lore** (`lore.txt`): chunking → Sentence Transformer embeddings → FAISS dense index, combined with BM25 lexical retrieval.
- **Structured facts** (`graph.trig`): RDF/SPARQL lookup through a dedicated tool.
- **Grounded generation**: a local Ollama model deciedes on the appropriate tool to call and answers only from retrieved context.

The retrieval stack is local.


In [2]:
# Run once in a fresh Colab/Jupyter environment.
%pip install -q langchain langchain-community langchain-ollama langgraph rdflib \
    sentence-transformers faiss-cpu rank_bm25 numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Start the local language model


In [3]:
%system apt-get update -qq && apt-get install -y -qq zstd

["W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)"]

In [4]:
%system curl -fsSL https://ollama.com/install.sh | sh

['>>> Cleaning up old version at /usr/local/lib/ollama',
 '>>> Installing ollama to /usr/local',
 '>>> Downloading ollama-linux-amd64.tar.zst',
 '#=#=#                                                                         ',
 '##O#-#                                                                        ',
 '##O=#  #                                                                      ',
 '',
 '                                                                           0.2%',
 '                                                                           0.7%',
 '#                                                                          1.6%',
 '##                                                                         3.5%',
 '###                                                                        5.0%',
 '####                                                                       6.4%',
 '#####                                                                      8.0%',
 '#######     

In [5]:
import platform
import shutil
import subprocess
import time
import requests

OLLAMA_MODEL = "mistral-nemo"

if shutil.which("ollama") is None:
    if platform.system() == "Linux":
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    else:
        raise RuntimeError("Install Ollama from https://ollama.com, then restart this notebook.")

def ensure_ollama_running():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2).raise_for_status()
        return
    except requests.RequestException:
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(5)

ensure_ollama_running()
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
print(f"Ollama is ready with {OLLAMA_MODEL}.")


Ollama is ready with mistral-nemo.


## 2. Load and chunk the corpus

In [6]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

LORE_PATH = Path("lore.txt")
GRAPH_PATH = Path("graph.trig")
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75
TOP_K = 4

if not LORE_PATH.exists():
    raise FileNotFoundError("Upload lore.txt to the notebook working directory before continuing.")

source_docs = TextLoader(str(LORE_PATH), encoding="utf-8").load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(source_docs)

for chunk_id, chunk in enumerate(chunks):
    chunk.metadata.update({"chunk_id": chunk_id, "source": LORE_PATH.name})

print({"source_documents": len(source_docs), "chunks": len(chunks)})


/tmp/ipykernel_2525/2861717502.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


{'source_documents': 1, 'chunks': 739}


## 3. Embed and index the chunks


In [7]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from langchain_community.retrievers import BM25Retriever

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMBEDDING_MODEL)
chunk_texts = [chunk.page_content for chunk in chunks]

chunk_embeddings = encoder.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

dense_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
dense_index.add(chunk_embeddings)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = min(12, len(chunks))

print({
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": chunk_embeddings.shape[1],
    "indexed_chunks": dense_index.ntotal,
})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dimension': 384, 'indexed_chunks': 739}


In [16]:
def dense_retrieve(query: str, candidate_k: int = 12):
    candidate_k = min(candidate_k, len(chunks))
    query_vector = encoder.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, positions = dense_index.search(query_vector, candidate_k)
    return [
        {"chunk_id": int(position), "dense_score": float(score)}
        for score, position in zip(scores[0], positions[0])
        if position >= 0
    ]


def hybrid_retrieve(query: str, top_k: int = TOP_K, candidate_k: int = 12):
    '''Fuse semantic and lexical rankings with Reciprocal Rank Fusion.'''
    candidate_k = min(candidate_k, len(chunks))
    dense_hits = dense_retrieve(query, candidate_k)
    bm25_retriever.k = candidate_k
    lexical_docs = bm25_retriever.invoke(query)

    lexical_ids = [int(doc.metadata["chunk_id"]) for doc in lexical_docs]
    dense_scores = {hit["chunk_id"]: hit["dense_score"] for hit in dense_hits}
    fused = {}
    rrf_constant = 60

    for rank, hit in enumerate(dense_hits, start=1):
        fused[hit["chunk_id"]] = fused.get(hit["chunk_id"], 0.0) + 1 / (rrf_constant + rank)
    for rank, chunk_id in enumerate(lexical_ids, start=1):
        fused[chunk_id] = fused.get(chunk_id, 0.0) + 1 / (rrf_constant + rank)

    ranked_ids = sorted(fused, key=fused.get, reverse=True)[:top_k]
    return [
        {
            "rank": rank,
            "chunk_id": chunk_id,
            "rrf_score": fused[chunk_id],
            "dense_score": dense_scores.get(chunk_id),
            "source": chunks[chunk_id].metadata.get("source", LORE_PATH.name),
            "text": chunks[chunk_id].page_content,
        }
        for rank, chunk_id in enumerate(ranked_ids, start=1)
    ]


def retrieval_table(query: str, top_k: int = TOP_K):
    '''Human-readable retrieval diagnostics.'''
    return pd.DataFrame(hybrid_retrieve(query, top_k=top_k))[
        ["rank", "chunk_id", "rrf_score", "dense_score", "source", "text"]
    ]


retrieval_table("Who killed a member of House Stark?", top_k=3)


,rank,chunk_id,rrf_score,dense_score,source,text
0,1,666,0.030536,0.479221,lore.txt,"Robb Stark, commonly known as The Young Wolf, ..."
1,2,356,0.016393,0.535155,lore.txt,Rickard Stark is a male character.\nHe is from...
2,3,224,0.016393,NaN,lore.txt,Jasper Waynwood is a male character.\nHe appea...


## 4. Expose hybrid retrieval and the RDF graph as tools

In [1]:
import rdflib
from langchain_core.tools import tool

@tool
def search_lore(query: str) -> str:
    '''Search biographies, narrative events and background lore with hybrid dense + BM25 retrieval.'''
    hits = hybrid_retrieve(query, top_k=TOP_K)
    if not hits:
        return "No relevant lore was found."
    return "\n\n".join(
        f"[lore:{hit['chunk_id']}] {hit['text']}" for hit in hits
    )


@tool
def query_graph_db(character_name: str) -> str:
    '''Look up structured relationships for a character: kills, family, allegiance, and marriage.'''
    if not GRAPH_PATH.exists():
        return "graph.trig is unavailable."

    graph = rdflib.Dataset(default_union=True)
    graph.parse(str(GRAPH_PATH), format="trig")
    normalized = character_name.strip().replace(" ", "_").lower()

    rows = []
    for subject, predicate, obj in graph.triples((None, None, None)):
        if normalized in str(subject).lower():
            pred = str(predicate).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
            value = str(obj).rsplit("/", 1)[-1].rsplit("#", 1)[-1].replace("_", " ")
            rows.append(f"- {pred}: {value}")
        if len(rows) >= 20:
            break

    if not rows:
        return f"No structured data found for {character_name}."
    return f"[graph:{character_name}]\n" + "\n".join(rows)


print("Hybrid lore and RDF graph tools are ready.")

ModuleNotFoundError: No module named 'rdflib'

## 5. Assistant


In [21]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
tools = [search_lore, query_graph_db]

class ListAnswer(BaseModel):
    items: list[str] = Field(
        description=(
            "Only the atomic values answering the user's question. "
            "Each item must be a name or short value copied from tool evidence. "
            "No explanations, Markdown or source references. "
            "Return 'No entity found.' when no answer exists."
        )
    )

fallback_formatter = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
).with_structured_output(
    ListAnswer,
    method="json_schema",
)

system_prompt = '''You are an expert question-answering assistant regarding Game of Thrones.
Decide whether to use "search_lore", "query_graph_db" or both, depending on the user's question. Use only retrieved evidence; never answer from model memory.

Tool routing examples:
- Parents, children, siblings, kills, service, marriage or guardianship:
  call query_graph_db(character_name="<character name>").
- Titles, aliases, biography, culture, appearances, birth or death:
  call search_lore(query="<complete user question>").
- Composite questions requiring both categories: call both tools.

Exact examples:
- query_graph_db(character_name="Jon Snow")
- search_lore(query="What titles does Jon Snow have?")

Never pass `query` to query_graph_db.
Never use query_graph_db to retrieve titles or aliases.
If a tool returns an invocation error, do not treat it as evidence.

CRITICAL REASONING RULE (AMNESIA):
You know absolutely NOTHING about Game of Thrones outside of what the tools return. If a user asks a question and the tools return 'No data found', you MUST reply: 'I do not have this information in my current database.'
DO NOT use pre-trained or external knowledge to answer. DO NOT invent relationship categories that are not explicitly printed by the 'query_graph_db' tool. Make a maximum of 5 attempt calls per tool. Formulate your final answer using ONLY the data you successfully retrieved, with no additional information.

STRICT OUTPUT CONTRACT:
- For user questions requesting one or more values, return a list of the retrieved values with items separated by ", ".
- The generated answer is direct, concise and does not provide additional details, introductions, conclusions, definitions or explanations. Do not justify your answer unless specifically asked (e.g., How did you derive this answer?).
- If the requested relationship is absent, return: 'No entity found in the database.'.
'''

agent_executor = create_agent(
    model=llm,
    tools=[search_lore, query_graph_db],
    system_prompt=system_prompt,
    response_format=ToolStrategy(
        ListAnswer,
        handle_errors="Return only the requested atomic values in `items`.",
    ),
)
print("The assistant is ready.")


The assistant is ready.


## 7. Ask questions

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage


NO_ANSWER = "I do not have this information in the provided context."


def render_structured_answer(result, user_query: str) -> str:
    """Render the agent answer, including composite questions."""
    structured = result.get("structured_response")

    if isinstance(structured, ListAnswer):
        raw_items = structured.items
    elif isinstance(structured, dict):
        raw_items = structured.get("items", [])
    else:
        raw_items = []

    items = []
    for item in raw_items or []:
        value = str(item).strip()
        if value and value not in items:
            items.append(value)

    if items:
        return ", ".join(items)

    # If the agent did not produce structured output, format all tool evidence.
    evidence = "\n\n".join(
        str(message.content).strip()
        for message in result.get("messages", [])
        if isinstance(message, ToolMessage)
        and str(message.content).strip()
    )

    if evidence:
        formatted = fallback_formatter.invoke(
            f"""
Answer every part of the composite question.

Rules:
- Use only the supplied tool evidence.
- Return all values requested by every part of the question.
- For example, include both parents and titles when both are requested.
- Return atomic values only.
- Do not include explanations, Markdown, tool names or introductions.
- Return an empty items list if the evidence does not contain the answers.

Question:
{user_query}

Evidence:
{evidence}
"""
        )

        formatted_items = []
        for item in formatted.items:
            value = item.strip()
            if value and value not in formatted_items:
                formatted_items.append(value)

        if formatted_items:
            return ", ".join(formatted_items)

    # Only inspect actual AI answers—not the original HumanMessage.
    for message in reversed(result.get("messages", [])):
        if not isinstance(message, AIMessage):
            continue

        content = message.content
        if isinstance(content, str) and content.strip():
            return content.strip()

    return NO_ANSWER


print("Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("\nUser: ").strip()

    if not user_input:
        continue
    if user_input.casefold() in {"exit", "quit"}:
        print("System shutting down. Valar Morghulis!")
        break

    try:
        result = agent_executor.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config={"recursion_limit": 12},
        )

        tool_names_by_id = {}

        for message in result.get("messages", []):
            if isinstance(message, AIMessage):
                for call in message.tool_calls or []:
                    tool_names_by_id[call.get("id")] = call.get(
                        "name",
                        "unknown_tool",
                    )

        for message in result.get("messages", []):
            if isinstance(message, ToolMessage):
                tool_name = (
                    getattr(message, "name", None)
                    or tool_names_by_id.get(message.tool_call_id)
                    or "unknown_tool"
                )
                preview = str(message.content).replace("\n", " ")
                print(f"Tool called: {tool_name}")
                print(f"Retrieved: {preview}")

        final_response = render_structured_answer(result, user_input)
        print(f"\nAssistant: {final_response}")

    except Exception as error:
        print(f"Request failed: {type(error).__name__}: {error}")


Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.

User: Who are Jon Snow's parents?
Tool called: query_graph_db
Retrieved: [graph:Jon Snow] - guardedBy: Eddard Stark - killed: Olly - killed: Alliser Thorne - killedBy: Bowen Marsh - killedBy: Othell Yarwyck - killed: Daenerys Targaryen - killed: Karl Tanner - childOf: Rhaegar Targaryen - guardedBy: Ghost - killed: Qhorin Halfhand - killed: White Walker - killed: Mance Rayder - killed: Janos Slynt - killed: Orell - killed: Bowen Marsh - killed: Styr - killed: Othell Yarwyck - marriedOrEngagedTo: Ygritte - childOf: Lyanna Stark - killed: Othor

Assistant: Rhaegar Targaryen, Lyanna Stark

User: What titles and aliases does Jon Snow have?
Tool called: search_lore
Retrieved: [lore:240] Jon Snow, commonly known as Lord Snow, is a male character. Other known aliases for Jon Snow include Ned Stark's Bastard, The Snow of Winterfell, The Crow-Come-Over, The 998th Lord Commander of the Night's Watch, The Bastard of Winte